In [1]:
"""Auxiliary EFG and nuclear quadrupole frequency utilities."""

from typing import Dict, Optional
import numpy as np
import numpy.typing as npt
from src.pcefg import constants

In [2]:
def Vzz_for_unit_charge_at_distance(r: float) -> float:
    """Calculate axial EFG component Vzz for a +1e point charge at distance r.

    Args:
        r: Distance from charge in meters.

    Returns:
        Vzz component in V/m^2.

    Raises:
        ValueError: If r <= 0.
    """
    if r <= 0:
        raise ValueError("Distance r must be strictly positive.")

    return (2.0 / (4.0 * np.pi * constants.EPSILON0)) * (
        constants.ELEMENTARY_CHARGE / (r**3)
    )


def gen_radial_EFG(
    charge_position: npt.ArrayLike,
    site_position: npt.ArrayLike,
    charge: float = constants.ELEMENTARY_CHARGE,
    Vzz: Optional[float] = None,
) -> npt.NDArray[np.float64]:
    """Construct point-charge EFG tensor at site_position generated by a source charge.

    Args:
        charge_position: Source charge location in meters, shape `(3,)`.
        site_position: Field evaluation position in meters, shape `(3,)`.
        charge: Source charge value in Coulombs (default: +1e).
        Vzz: Optional direct axial EFG value in V/m^2. If None, calculated from point model.

    Returns:
        Symmetric 3x3 EFG tensor in V/m^2.

    Raises:
        ValueError: If charge_position and site_position coincide.
    """
    pos_src = np.asarray(charge_position, dtype=np.float64)
    pos_eval = np.asarray(site_position, dtype=np.float64)
    x = pos_eval - pos_src
    r = float(np.linalg.norm(x))

    if r <= 0:
        raise ValueError("Evaluation site and charge position cannot coincide.")

    x_hat = x / r
    if Vzz is None:
        Vzz = (charge / constants.ELEMENTARY_CHARGE) * Vzz_for_unit_charge_at_distance(r)

    return 0.5 * Vzz * (3.0 * np.outer(x_hat, x_hat) - np.eye(3))


def get_omegaQ_mu(
    I: float,
    Q: float,
    r: float,
    gamma_sternheimer: float = 0.0,
) -> float:
    """Compute muon-induced quadrupole interaction angular frequency $\\omega_{Q,\\mu}$.

    Args:
        I: Nuclear spin quantum number ($I > 1/2$).
        Q: Electric quadrupole moment in m^2.
        r: Muon-nucleus distance in meters.
        gamma_sternheimer: Sternheimer antishielding factor.

    Returns:
        Angular quadrupole frequency $\\omega_{Q,\\mu}$ in rad/s.
    """
    if I <= 0.5:
        return 0.0

    prefactor = (
        -(1.0 - gamma_sternheimer)
        * (1.0 / constants.HBAR)
        * (1.0 / (4.0 * np.pi * constants.EPSILON0))
    )
    numerator = 3.0 * (constants.ELEMENTARY_CHARGE**2) * Q
    denominator = 2.0 * I * (2.0 * I - 1.0) * (r**3)
    return prefactor * (numerator / denominator)


def EFG_from_omegaq_PAS(
    omegaq: float,
    eta: float,
    m: float,
    I: float,
    Q: float,
) -> npt.NDArray[np.float64]:
    """Reconstruct 3x3 EFG tensor in its Principal Axis System (PAS) from $\\omega_Q$.

    Args:
        omegaq: Quadrupole transition frequency in rad/s.
        eta: EFG asymmetry parameter ($0 \\le \\eta \\le 1$).
        m: Magnetic quantum number $m$.
        I: Nuclear spin quantum number.
        Q: Electric quadrupole moment in m^2.

    Returns:
        Diagonalized 3x3 EFG tensor in V/m^2.
    """
    a_factor = omegaq / (3.0 * (2.0 * np.abs(m) + 1.0) / constants.HBAR)
    v_zz = a_factor * (4.0 * I * (2.0 * I - 1.0)) / (Q * constants.ELEMENTARY_CHARGE)

    v_xx = +0.5 * v_zz * (eta - 1.0)
    v_yy = -0.5 * v_zz * (eta + 1.0)

    return np.diag([v_xx, v_yy, v_zz])


def nu_Q(I: float, Q: float, Vzz: float) -> float:
    """Calculate fundamental quadrupole frequency $\\nu_Q$ in MHz.

    Args:
        I: Nuclear spin quantum number.
        Q: Electric quadrupole moment in m^2.
        Vzz: Principal EFG component in V/m^2.

    Returns:
        Frequency $\\nu_Q$ in MHz.
    """
    if I <= 0.5:
        return 0.0

    return (
        (3.0 * constants.ELEMENTARY_CHARGE * Q * Vzz)
        / (2.0 * I * (2.0 * I - 1.0) * constants.H_PLANCK)
        * 1e-6
    )


def quadrupole_frequencies(
    I: float,
    Q: float,
    Vzz: float,
    eta: float,
) -> Dict[str, float]:
    """Compute directional quadrupole frequency components and $\\nu_Q$ in MHz.

    Args:
        I: Nuclear spin quantum number.
        Q: Electric quadrupole moment in m^2.
        Vzz: Principal EFG component $V_{zz}$ in V/m^2.
        eta: EFG asymmetry parameter $\\eta$.

    Returns:
        Dictionary containing `Vxx`, `Vyy`, `Vzz`, `eta`, `nu_Q_MHz`, `nu_x_MHz`,
        `nu_y_MHz`, and `nu_z_MHz`.
    """
    if I <= 0.5:
        return {
            "Vxx": 0.0,
            "Vyy": 0.0,
            "Vzz": Vzz,
            "eta": eta,
            "nu_Q_MHz": 0.0,
            "nu_x_MHz": 0.0,
            "nu_y_MHz": 0.0,
            "nu_z_MHz": 0.0,
        }

    prefactor = (
        (3.0 * constants.ELEMENTARY_CHARGE * Q)
        / (2.0 * I * (2.0 * I - 1.0) * constants.H_PLANCK)
        * 1e-6
    )

    v_xx = +0.5 * Vzz * (eta - 1.0)
    v_yy = -0.5 * Vzz * (eta + 1.0)

    nu_x_mhz = prefactor * v_xx
    nu_y_mhz = prefactor * v_yy
    nu_z_mhz = prefactor * Vzz
    nu_q_mhz = abs(nu_z_mhz * np.sqrt(1.0 + (eta**2) / 3.0))

    return {
        "Vxx": v_xx,
        "Vyy": v_yy,
        "Vzz": Vzz,
        "eta": eta,
        "nu_Q_MHz": nu_q_mhz,
        "nu_x_MHz": nu_x_mhz,
        "nu_y_MHz": nu_y_mhz,
        "nu_z_MHz": nu_z_mhz,
    }